# Integrative Analysis of Breast Cancer Susceptibility Loci and Tumor Immune Programs Reveals Immune-Associated Candidate Genes in Triple-Negative Breast Cancer

---
## Notebook 4 - Differential Gene Expression Analysis Identifies the TNBC Transcriptional Landscape


### Overview

This notebook identifies genes that are differentially expressed between triple-negative breast cancer (TNBC) and other breast cancer subtypes.

Using raw RNA-seq count data and the DESeq2 statistical framework, gene expression levels are compared between the two clinical groups while accounting for differences in sequencing depth and biological variability.

The resulting differentially expressed genes (DEGs) define the TNBC transcriptional landscape and provide the molecular foundation for subsequent pathway enrichment and integrative analyses with GWAS-derived susceptibility genes.

In [1]:
import pandas as pd
import numpy as np
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

### 1. Load Processed Transcriptomic and Clinical Data

In [2]:
counts = pd.read_csv("../data/rna_seq/brca_counts_cleaned.csv", index_col=0)
metadata = pd.read_csv("../data/rna_seq/brca_clinical_cleaned.csv", index_col=0)

### 2. Prepare the Count Matrix for Differential Expression Analysis

DESeq2 expects a matrix with samples represented as rows and genes represented as columns.

The raw count matrix is therefore transposed and converted to integer counts, preserving the original sequencing counts required by the negative binomial statistical model.

In [3]:
counts_inverted = counts.T

counts_inverted = counts_inverted.astype(int)

print(f"Counts shape for PyDESeq2 (Samples x Genes): {counts_inverted.shape}")
print(f"Metadata shape for PyDESeq2 (Samples x Design Factors): {metadata.shape}")

Counts shape for PyDESeq2 (Samples x Genes): (725, 19969)
Metadata shape for PyDESeq2 (Samples x Design Factors): (725, 6)


### 3. Filter Lowly Expressed Genes

Genes with extremely low read counts provide little statistical power and can increase the multiple-testing burden.

Genes expressed at fewer than 10 counts in at least 10% of samples are removed prior to model fitting.

This filtering step improves statistical stability while retaining biologically informative genes.

In [4]:
min_samples = int(0.10 * counts_inverted.shape[0])
keep_genes = (counts_inverted >= 10).sum(axis=0) >= min_samples
counts_inverted_filtered = counts_inverted.loc[:, keep_genes]

counts_inverted_filtered = counts_inverted_filtered.astype(int)

print(f"Filtered matrix ready for DESeq2: {counts_inverted_filtered.shape}")

Filtered matrix ready for DESeq2: (725, 16791)


### 4. Verify Gene Identifier Uniqueness

Each gene must be represented only once within the expression matrix.

Duplicate gene names are identified and removed to ensure that every statistical test corresponds to a unique transcriptomic feature.

In [5]:
if counts_inverted_filtered.columns.duplicated().any():
    print(f"Found {counts_inverted_filtered.columns.duplicated().sum()} duplicate gene names! Removing...")
    counts_inverted_filtered = counts_inverted_filtered.loc[:, ~counts_inverted_filtered.columns.duplicated(keep='first')]

print("Verification: Are gene labels unique now?", counts_inverted_filtered.columns.is_unique)

Found 3 duplicate gene names! Removing...
Verification: Are gene labels unique now? True


### 5. Model Gene Expression Using DESeq2

Differential expression analysis is performed using DESeq2, which models RNA-seq count data with negative binomial generalized linear models.

The experimental design compares:

**TNBC vs Other Breast Cancer**

During model fitting, DESeq2 estimates:

- size factors for library normalization
- gene-wise dispersion parameters
- empirical Bayes dispersion shrinkage
- log2 fold changes
- Wald test statistics

In [6]:
print("\nInitializing PyDESeq2 DataSet object...")
dds = DeseqDataSet(
    counts=counts_inverted_filtered,
    metadata=metadata,
    design="~condition_group",
    ref_level=["condition_group", "Other_Breast_Cancer"],
    n_cpus=4 
)

print("Fitting Generalized Linear Models (this may take a few minutes)...")
dds.deseq2()


Initializing PyDESeq2 DataSet object...


C:\Users\Nonye\AppData\Local\Temp\ipykernel_11044\1393421323.py:2: DeprecationWarning: ref_level is deprecated and no longer has any effect. It will beremoved in a future release.
  dds = DeseqDataSet(
Fitting size factors...


Fitting Generalized Linear Models (this may take a few minutes)...
Using None as control genes, passed at DeseqDataSet initialization


... done in 0.72 seconds.

Fitting dispersions...
... done in 41.18 seconds.

Fitting dispersion trend curve...
... done in 1.78 seconds.

Fitting MAP dispersions...
... done in 37.95 seconds.

Fitting LFCs...
... done in 14.12 seconds.

Calculating cook's distance...
... done in 2.08 seconds.

Replacing 1695 outlier genes.

Fitting dispersions...
... done in 3.05 seconds.

Fitting MAP dispersions...
... done in 2.83 seconds.

Fitting LFCs...
... done in 2.16 seconds.



### 6. Identify Differentially Expressed Genes

Wald tests are performed to determine whether each gene is significantly differentially expressed between TNBC and the comparison cohort.

P-values are adjusted using the Benjamini–Hochberg False Discovery Rate (FDR) procedure to control for multiple hypothesis testing.

In [7]:
stat_res = DeseqStats(dds, contrast=["condition_group", "TNBC", "Other_Breast_Cancer"], n_cpus=4)

stat_res.summary()

results_df = stat_res.results_df
results_df = results_df.dropna(subset=["pvalue", "padj"]) 

results_df.index.name = "gene_name"

Running Wald tests...
... done in 3.93 seconds.



Log2 fold change & Wald test p-value: condition_group TNBC vs Other_Breast_Cancer
                 baseMean  log2FoldChange     lfcSE      stat        pvalue  \
gene_name                                                                     
TSPAN6        3066.838720        0.445756  0.096995  4.595644  4.314149e-06   
TNMD            38.020886       -0.329402  0.218499 -1.507568  1.316650e-01   
DPM1          2384.319349        0.170410  0.057653  2.955788  3.118718e-03   
SCYL3         1626.930919       -0.330910  0.055459 -5.966722  2.420675e-09   
C1orf112       739.872272        0.697587  0.071130  9.807156  1.048827e-22   
...                   ...             ...       ...       ...           ...   
DUS4L-BCAP29   309.000338       -0.018968  0.066859 -0.283696  7.766434e-01   
NPBWR1           8.701117        1.948952  0.251311  7.755140  8.824583e-15   
AC010980.1      24.593976        1.179237  0.218322  5.401357  6.613859e-08   
AL391628.1       7.747052       -0.084010  0.1034

### 7. Rank and Annotate Differentially Expressed Genes

Each gene is classified into one of three categories:

- Upregulated
- Downregulated
- Not Significant

The complete results table is ranked according to statistical significance and expression change to facilitate downstream biological interpretation.

In [8]:
results_df["Significance"] = "Not Significant"

is_up = (results_df["padj"] < 0.05) & (results_df["log2FoldChange"] > 1)
is_down = (results_df["padj"] < 0.05) & (results_df["log2FoldChange"] < -1)

results_df.loc[is_up, "Significance"] = "Upregulated"
results_df.loc[is_down, "Significance"] = "Downregulated"

results_df["Significance"] = pd.Categorical(
    results_df["Significance"], 
    categories=["Upregulated", "Downregulated", "Not Significant"],
    ordered=True
)

ranked_result = results_df.sort_values(by=["Significance", "padj"], ascending=[True, True])

print(f"Total pure verified high-contrast DEGs exported: {ranked_result.shape[0]}")
print(ranked_result["Significance"].value_counts())

Total pure verified high-contrast DEGs exported: 16788
Significance
Not Significant    13841
Upregulated         1602
Downregulated       1345
Name: count, dtype: int64


### 8. Export Differential Expression Results

In [9]:
dysregulated_result = ranked_result[ranked_result["Significance"] != "Not Significant" ].copy()

final_column_layout = [
     "Significance", 
    "log2FoldChange", "pvalue", "padj"
]
dysregulated_result = dysregulated_result [final_column_layout]

In [10]:
ranked_result.to_csv("../results/tables/brca_tnbc_compiled_degs.csv")
dysregulated_result.to_csv("../results/tables/dysregulated_brca_tnbc_compiled_degs.csv")

### Biological Summary

Differential expression analysis identified substantial transcriptional differences between TNBC and other breast cancer subtypes.

Approximately three thousand genes satisfied the predefined significance criteria, indicating widespread molecular reprogramming within TNBC tumors.

These differentially expressed genes represent candidate biological processes associated with tumor aggressiveness, proliferation, immune activity, metabolism, and stromal remodeling. Subsequent notebooks investigate whether inherited breast cancer susceptibility genes are enriched within these transcriptional alterations.